# Minimal Statistical Test

Only the essentials:
1. Set `DATASETS` (two files)
2. Set `METRIC_KEY`
3. Run the notebook


In [4]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from TSB_AD.evaluation.metrics import get_metrics 
from TSB_AD.model_wrapper import run_Unsupervise_AD 

warnings.filterwarnings("ignore")

## Configuration 

In [5]:
METHOD_NAME = "Sub_PCA"
DATASET_NAME = "TODS"

DATASET_DIRECTORY = Path("../datasets/TSB-AD-U")
DATASET_IDS = sorted({int(file.name.split('_')[0]) for file in DATASET_DIRECTORY.glob(f'*{DATASET_NAME}*.csv')})

DATASETS = [
    file 
    for dataset_id in DATASET_IDS 
    for file in DATASET_DIRECTORY.glob(f"{dataset_id:03d}_{DATASET_NAME}_id_*.csv")
]
DATASETS = sorted(DATASETS, key=lambda x: int(x.name.split('_')[0]))

METRIC_KEY = "AUC-ROC"  # "AUC-PR", "PA-F1", "VUS-PR", "VUS-ROC"

print(f"Method: {METHOD_NAME}")
print(f"Datasets: {[p.name for p in DATASETS]}")
print(f"Metric: {METRIC_KEY}")

Method: Sub_PCA
Datasets: ['287_TODS_id_1_Synthetic_tr_500_1st_11.csv', '288_TODS_id_2_Synthetic_tr_500_1st_65.csv', '289_TODS_id_3_Synthetic_tr_500_1st_26.csv', '290_TODS_id_4_Synthetic_tr_500_1st_26.csv', '291_TODS_id_5_Synthetic_tr_500_1st_11.csv', '292_TODS_id_6_Synthetic_tr_500_1st_11.csv', '293_TODS_id_7_Synthetic_tr_500_1st_7.csv', '294_TODS_id_8_Synthetic_tr_500_1st_200.csv', '295_TODS_id_9_Synthetic_tr_1250_1st_2046.csv', '296_TODS_id_10_Synthetic_tr_500_1st_26.csv', '297_TODS_id_11_Synthetic_tr_500_1st_7.csv', '298_TODS_id_12_Synthetic_tr_500_1st_0.csv', '299_TODS_id_13_Synthetic_tr_500_1st_65.csv', '300_TODS_id_14_Synthetic_tr_1250_1st_2555.csv', '301_TODS_id_15_Synthetic_tr_500_1st_245.csv']
Metric: AUC-ROC


In [6]:
rows = []

# for dataset_path in DATASETS:
dataset_path = DATASETS[0]
df = pd.read_csv(dataset_path).dropna()
data = df.iloc[:, :-1].values.astype(float)
labels = df["Label"].astype(int).to_numpy()

# if return is str -> error
scores = run_Unsupervise_AD(METHOD_NAME, data, periodicity=1)
if isinstance(scores, str):
    raise RuntimeError(scores)

print(f"Shape of scores: {scores.shape}, Shape of labels: {labels.shape}")
# ravel -> to 1D array (important when AD on multivariate data)
scores = np.asarray(scores)
print(f"Shape of scores after conversion: {scores.shape}")
scores = scores.ravel()
print(f"Shape of scores after ravel: {scores.shape}")
# Ensure scores and labels have the same length
n = min(len(scores), len(labels))
scores = scores[:n]
labels = labels[:n]

metrics = get_metrics(scores, labels)
# Check if my selected metric is in the returned metrics
if METRIC_KEY not in metrics:
    raise ValueError(f"Metric '{METRIC_KEY}' not found. Available: {list(metrics.keys())}")

rows.append(
    {
        "dataset": dataset_path.name,
        "method": METHOD_NAME,
        METRIC_KEY: float(metrics[METRIC_KEY]),
    }
)

results_df = pd.DataFrame(rows)
display(results_df)

Shape of scores: (5000,), Shape of labels: (5000,)
Shape of scores after conversion: (5000,)
Shape of scores after ravel: (5000,)


,dataset,method,AUC-ROC
0,287_TODS_id_1_Synthetic_tr_500_1st_11.csv,Sub_PCA,0.628449


In [7]:
print(f"Mean {METRIC_KEY}: {results_df[METRIC_KEY].mean():.4f}")

Mean AUC-ROC: 0.6284


### Helper functions from TSB-AD repo
https://github.com/thedatumorg/TSB-AD

In [8]:
from statsmodels.tsa.stattools import acf
from scipy.signal import argrelextrema
import numpy as np
from statsmodels.graphics.tsaplots import plot_acf

# determine sliding window (period) based on ACF


def find_length_rank(data, rank=1):
    data = data.squeeze()
    if len(data.shape) > 1:
        return 0
    if rank == 0:
        return 1
    data = data[:min(20000, len(data))]

    base = 3
    auto_corr = acf(data, nlags=400, fft=True)[base:]

    # plot_acf(data, lags=400, fft=True)
    # plt.xlabel('Lags')
    # plt.ylabel('Autocorrelation')
    # plt.title('Autocorrelation Function (ACF)')
    # plt.savefig('/data/liuqinghua/code/ts/TSAD-AutoML/AutoAD_Solution/candidate_pool/cd_diagram/ts_acf.png')

    local_max = argrelextrema(auto_corr, np.greater)[0]

    # print('auto_corr: ', auto_corr)
    # print('local_max: ', local_max)

    try:
        # max_local_max = np.argmax([auto_corr[lcm] for lcm in local_max])
        sorted_local_max = np.argsort([auto_corr[lcm] for lcm in local_max])[
            ::-1]    # Ascending order
        max_local_max = sorted_local_max[0]     # Default
        if rank == 1:
            max_local_max = sorted_local_max[0]
        if rank == 2:
            for i in sorted_local_max[1:]:
                if i > sorted_local_max[0]:
                    max_local_max = i
                    break
        if rank == 3:
            for i in sorted_local_max[1:]:
                if i > sorted_local_max[0]:
                    id_tmp = i
                    break
            for i in sorted_local_max[id_tmp:]:
                if i > sorted_local_max[id_tmp]:
                    max_local_max = i
                    break
        # print('sorted_local_max: ', sorted_local_max)
        # print('max_local_max: ', max_local_max)

        final_period = local_max[max_local_max]+base
        if final_period > 300:
            return 125
        return final_period
    except:
        return 125


# determine sliding window (period) based on ACF, Original version
def find_length(data):
    if len(data.shape) > 1:
        return 0
    data = data[:min(20000, len(data))]

    base = 3
    auto_corr = acf(data, nlags=400, fft=True)[base:]

    local_max = argrelextrema(auto_corr, np.greater)[0]
    try:
        max_local_max = np.argmax([auto_corr[lcm] for lcm in local_max])
        final_period = local_max[max_local_max]+base
        if final_period > 300:
            return 125
        return final_period
    except:
        return 125

In [9]:
# Run PCA manually and inspect which points were flagged as anomalies

from TSB_AD.models.PCA import PCA

periodicity = 1
sliding_window = find_length_rank(data, rank=periodicity)
print(f"sliding_window: {sliding_window}")
n_components = None

clf = PCA(slidingWindow=sliding_window, n_components=n_components)
clf.fit(data)

scores = np.asarray(clf.decision_scores_).ravel()
predicted_labels = np.asarray(clf.labels_).astype(int).ravel()
ground_truth_labels = np.asarray(labels).astype(int).ravel()

# get the minimum length among scores, predicted_labels, ground_truth_labels
if len(scores) != len(predicted_labels) or len(scores) != len(ground_truth_labels):
    print(
        f"Warning: Length mismatch - scores: {len(scores)}, predicted_labels: {len(predicted_labels)}, ground_truth_labels: {len(ground_truth_labels)}")
n = min(len(scores), len(predicted_labels), len(ground_truth_labels))
scores = scores[:n]
predicted_labels = predicted_labels[:n]
ground_truth_labels = ground_truth_labels[:n]

# predicted and ground-truth anomalies
predicted_anomaly_idx = np.flatnonzero(predicted_labels)
ground_truth_anomaly_idx = np.flatnonzero(ground_truth_labels)
# TP, FP, FN
true_positive_idx = np.intersect1d(
    predicted_anomaly_idx, ground_truth_anomaly_idx)
false_positive_idx = np.setdiff1d(
    predicted_anomaly_idx, ground_truth_anomaly_idx)
false_negative_idx = np.setdiff1d(
    ground_truth_anomaly_idx, predicted_anomaly_idx)

comparison_df = pd.DataFrame(
    {
        "score": scores,
        "predicted_label": predicted_labels,
        "ground_truth_label": ground_truth_labels,
    }
)
# comparison_df.index.name = "index"
# comparison_df = comparison_df.reset_index()
comparison_df["match_type"] = "true_negative"
comparison_df.loc[
    (comparison_df["predicted_label"] == 1) & (
        comparison_df["ground_truth_label"] == 1),
    "match_type",
] = "true_positive"
comparison_df.loc[
    (comparison_df["predicted_label"] == 1) & (
        comparison_df["ground_truth_label"] == 0),
    "match_type",
] = "false_positive"
comparison_df.loc[
    (comparison_df["predicted_label"] == 0) & (
        comparison_df["ground_truth_label"] == 1),
    "match_type",
] = "false_negative"

preview_df = comparison_df[comparison_df["match_type"]
                           != "true_negative"].head(20)

print(f"Dataset: {dataset_path.name}")
print(
    f"PCA parameters: sliding_window={sliding_window}, periodicity={periodicity}, n_components={n_components}")
print(f"Aligned length: {n}")
print(f"Predicted anomalies: {len(predicted_anomaly_idx)}")
print(f"Ground-truth anomalies: {len(ground_truth_anomaly_idx)}")
print(f"True positives: {len(true_positive_idx)}")
print(f"False positives: {len(false_positive_idx)}")
print(f"False negatives: {len(false_negative_idx)}")

display(preview_df)

sliding_window: 100
Dataset: 287_TODS_id_1_Synthetic_tr_500_1st_11.csv
PCA parameters: sliding_window=100, periodicity=1, n_components=None
Aligned length: 5000
Predicted anomalies: 500
Ground-truth anomalies: 261
True positives: 48
False positives: 452
False negatives: 213


,score,predicted_label,ground_truth_label,match_type
11,3.687813e+06,0,1,false_negative
12,3.687813e+06,0,1,false_negative
27,3.687813e+06,0,1,false_negative
62,3.688007e+06,0,1,false_negative
68,3.683528e+06,0,1,false_negative
111,3.684192e+06,0,1,false_negative
116,3.665703e+06,0,1,false_negative
185,3.685717e+06,0,1,false_negative
194,3.706740e+06,0,1,false_negative
209,3.709399e+06,0,1,false_negative


In [10]:
# Plot all data in disjoint windows and save to disk

from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to prevent memory leaks

WINDOW_SIZE = 200

# Save plots in results/error_windows/<method>/<dataset>/
SAVE_PLOTS = True
PLOT_DPI = 150
PLOT_OUTPUT_DIR = Path("../results") / "windows" / \
    METHOD_NAME / dataset_path.stem
if SAVE_PLOTS:
    PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

series = np.asarray(data).squeeze()[:n]
if series.ndim != 1:
    raise ValueError(
        "This plotting cell expects univariate data after squeeze().")

print(f"Dataset: {dataset_path.name}")
print(f"Window size: {WINDOW_SIZE}")
if SAVE_PLOTS:
    print(f"Plots will be saved to: {PLOT_OUTPUT_DIR.resolve()}")

for start_idx in range(0, n, WINDOW_SIZE):
    end_idx = min(start_idx + WINDOW_SIZE, n)

    window_indices = np.arange(start_idx, end_idx)
    window_values = series[start_idx:end_idx]
    window_gt = ground_truth_labels[start_idx:end_idx]

    window_fp = np.intersect1d(window_indices, false_positive_idx)
    window_fn = np.intersect1d(window_indices, false_negative_idx)
    window_tp = np.intersect1d(window_indices, true_positive_idx)

    fig, ax = plt.subplots(figsize=(14, 4))

    normal_mask = window_gt == 0
    anomaly_mask = window_gt == 1

    ax.plot(window_indices, window_values, color="gray",
            alpha=0.3, linewidth=1, zorder=0)
    ax.plot(window_indices[normal_mask], window_values[normal_mask],
            "o", color="blue", markersize=3, alpha=0.7)
    ax.plot(window_indices[anomaly_mask], window_values[anomaly_mask],
            "o", color="red", markersize=6, alpha=0.7)

    if len(window_tp) > 0:
        tp_values = series[window_tp]
        # We mark TPs with a distinct marker (e.g., green square border) around the red anomaly
        ax.plot(window_tp, tp_values, "s", color="green", markersize=8,
                markeredgewidth=1.5, markerfacecolor="none")
    if len(window_fp) > 0:
        fp_values = series[window_fp]
        ax.plot(window_fp, fp_values, "x", color="orange",
                markersize=8, markeredgewidth=1.5)
    if len(window_fn) > 0:
        fn_values = series[window_fn]
        ax.plot(window_fn, fn_values, "x", color="black",
                markersize=8, markeredgewidth=1.5)

    ax.set_xlabel("Time Index", fontsize=12)
    ax.set_ylabel("Value", fontsize=12)
    ax.set_title(
        f"{dataset_path.name}: Window {start_idx}-{end_idx - 1}",
        fontsize=14,
    )

    legend_elements = [
        Patch(facecolor="blue", label="Normal"),
        Patch(facecolor="red", label="Ground-Truth Anomaly"),
        Patch(facecolor="green", label="True Positive"),
        Patch(facecolor="orange", label="False Positive"),
        Patch(facecolor="black", label="False Negative"),
    ]
    ax.legend(handles=legend_elements, loc="upper right")

    plt.tight_layout()

    if SAVE_PLOTS:
        plot_filename = f"window_{start_idx:05d}_{end_idx:05d}.png"
        plot_path = PLOT_OUTPUT_DIR / plot_filename
        fig.savefig(plot_path, dpi=PLOT_DPI, bbox_inches="tight")

    plt.close(fig)

print("Finished generating window plots.")

Dataset: 287_TODS_id_1_Synthetic_tr_500_1st_11.csv
Window size: 200
Plots will be saved to: /home/lukas/Code/Master/tsb-ad-test/results/windows/Sub_PCA/287_TODS_id_1_Synthetic_tr_500_1st_11
Finished generating window plots.
